# 00 — Environment and CUDA check

**Purpose:** Verify the active Conda kernel, Windows resources, PyTorch/CUDA, packages, disk and writable outputs.

**Required inputs:** `config.local.yaml`; installed Python environment. No dataset is needed.

**Saved outputs:** `outputs/environment_report.json`, `outputs/environment_packages.txt`.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [5]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

Project root: D:\CellCancer\leukemia-yolo
Configuration: D:\CellCancer\leukemia-yolo\config.local.yaml


## 2. Inspect environment

PASS/WARNING/FAIL are colored. System RAM and NVIDIA dedicated VRAM are separate fields.

In [6]:
from src.environment import inspect_environment, status_table
report = inspect_environment(cfg)
table = status_table(report)
display(table[["check", "value", "status", "detail"]])

,check,value,status,detail
0,Python executable,c:\Users\majid\.conda\envs\leukemia-yolo\python.exe,PASS,
1,Python version,3.11.17,PASS,
2,Conda environment and active kernel,Not set,WARNING,Kernel executable must belong to leukemia-yolo; never use base for training.
3,Operating system,Windows 10 10.0.26200,PASS,
4,System RAM total / available GiB,15.73 / 1.86,PASS,
5,Dataset-drive free GiB,140.11,PASS,Check in Windows Optimize Drives that this drive is SSD before choosing disk cache.
6,Dataset-drive media type,SSD,PASS,"If unknown, verify in Windows Optimize Drives. Disk cache should use an SSD with enough free space."
7,Storage type,Fixed drive,WARNING,Confirm SSD manually in Windows Optimize Drives before disk caching.
8,PyTorch,2.14.1+cu126,PASS,
9,Torchvision,0.29.1+cu126,PASS,


,check,value,status,detail
0,Python executable,c:\Users\majid\.conda\envs\leukemia-yolo\pytho...,PASS,
1,Python version,3.11.17,PASS,
2,Conda environment and active kernel,Not set,WARNING,Kernel executable must belong to leukemia-yolo...
3,Operating system,Windows 10 10.0.26200,PASS,
4,System RAM total / available GiB,15.73 / 1.86,PASS,
5,Dataset-drive free GiB,140.11,PASS,Check in Windows Optimize Drives that this dri...
6,Dataset-drive media type,SSD,PASS,"If unknown, verify in Windows Optimize Drives...."
7,Storage type,Fixed drive,WARNING,Confirm SSD manually in Windows Optimize Drive...
8,PyTorch,2.14.1+cu126,PASS,
9,Torchvision,0.29.1+cu126,PASS,


## 3. Verify GPU gate

A CPU-only wheel or unavailable driver produces an explicit failure. Training also checks this gate in its child process.

In [7]:
from src.environment import require_cuda
try:
    gpu = require_cuda(cfg["device"])
    print("PASS — CUDA GPU:", gpu)
except RuntimeError as exc:
    print("FAIL — CUDA training unavailable:", exc)

PASS — CUDA GPU: {'gpu_name': 'NVIDIA GeForce RTX 3050 Ti Laptop GPU', 'dedicated_vram_gib': 3.99951171875, 'available_vram_gib': 3.212597656995058, 'compute_capability': '8.6', 'torch_version': '2.14.1+cu126', 'torch_cuda_version': '12.6', 'device': 0}


## 4. Saved report and status

Reports are generated even when CUDA is absent, so the development laptop can complete this notebook.

In [8]:
assert (out / "environment_report.json").is_file()
assert (out / "environment_packages.txt").is_file()
print("PASS — environment report saved" if report["cuda_ready"] and report["kernel_ready"] else
      "WARNING — report saved; resolve FAIL/WARNING rows before GPU training")

WARNING — report saved; resolve FAIL/WARNING rows before GPU training


## Next notebook

Open **01_BCCD_Dataset_Inspection.ipynb** after setting the dataset path in `config.local.yaml`.